# Image Motion Studio — CPU Worker (no GPU needed)

This notebook runs the studio worker loop on **any machine — CPU only**. It loads no model itself: it claims jobs from the studio and delegates real WAN image-to-video inference to a free **Hugging Face ZeroGPU Space** that runs WAN 2.2, then uploads the finished MP4 into your Library.

**What you need:**

1. A Hugging Face Space that runs WAN image-to-video (a public one, or your own duplicate). The default assumes an API endpoint named `/infer`.
2. Studio values: `CONVEX_HTTP_BASE` + `WORKER_TOKEN` (Settings → worker setup).
3. Optionally a free HF token (`hf_...`) — needed only for gated/queued Spaces.

Run it in Colab (CPU runtime), locally, or anywhere with internet. Keep the last cell running — it polls for jobs just like the GPU worker.

In [ ]:
#@title 1 · Configuration — paste your studio values

# From the studio Settings page → worker setup:
CONVEX_HTTP_BASE = ""  #@param {type:"string"}
WORKER_TOKEN = ""             #@param {type:"string"}

# The Hugging Face Space that performs the actual WAN image-to-video
# inference on its ZeroGPU. Paste "owner/space-name".
HF_SPACE_ID = ""  #@param {type:"string"}
# The Space API endpoint name (visible on the Space under "API" / via view_api()).
SPACE_API_NAME = "/infer"  #@param {type:"string"}

# Optional free HF token (hf_...) — needed only if the Space is gated or the
# ZeroGPU queue requires authentication. Leave empty for public Spaces.
HF_TOKEN = ""  #@param {type:"string"}

WORKER_VERSION = "1.1.0-cpu"

# ---- Configuration validation: fail loudly and early ----
errors = []
if not CONVEX_HTTP_BASE.startswith("https://"):
    errors.append("CONVEX_HTTP_BASE must be your .convex.site URL (starts with https://). Copy it from the studio Settings page.")
if len(WORKER_TOKEN) < 16:
    errors.append("WORKER_TOKEN is missing or too short. Generate one in the studio (Settings → Worker token → Generate token) and paste it here.")
if "/" not in HF_SPACE_ID or HF_SPACE_ID.startswith("/"):
    errors.append('HF_SPACE_ID must look like "owner/space-name" (the Space that runs WAN image-to-video).')
if errors:
    print("=" * 60)
    for e in errors:
        print("CONFIG ERROR:", e)
    print("=" * 60)
    raise SystemExit(1)

import requests, json, time

def api(path, payload):
    r = requests.post(f"{CONVEX_HTTP_BASE}/worker_api/{path}", json={"token": WORKER_TOKEN, **payload}, timeout=120)
    r.raise_for_status()
    return r.json()

# First contact — a wrong token or URL fails here, loudly.
health = api("health", {"status": "online", "workerVersion": WORKER_VERSION, "message": "CPU Space worker starting"})
print("✓ authenticated —", CONVEX_HTTP_BASE, json.dumps(health))

In [ ]:
#@title 2 · Worker identity — register as CPU worker
# This worker has no local GPU: inference happens remotely on the HF Space.
WORKER_LABEL = f"HF Space · {HF_SPACE_ID}"

api("health", {
    "status": "online",
    "gpuName": WORKER_LABEL,
    "workerVersion": WORKER_VERSION,
    "message": "CPU worker — inference via HF Space",
})
print("✓ registered as CPU worker:", WORKER_LABEL)
if HF_TOKEN:
    print("· HF token provided (gated/queued Space access)")
else:
    print("· no HF token — assuming a public Space (add one if connection is denied)")

In [ ]:
#@title 3 · Install client dependencies (~30 seconds)
# gradio_client talks to HF Spaces. No torch / no model download here.
%pip -q install -U gradio_client
import gradio_client
print("gradio_client installed")

In [ ]:
#@title 4 · Connect to the inference Space
from gradio_client import Client, handle_file

print(f"· connecting to Space {HF_SPACE_ID} …")
client = Client(HF_SPACE_ID, hf_token=HF_TOKEN or None)

print("✓ connected. API endpoints exposed by this Space:")
client.view_api(all_endpoints=True)

print(f"· using endpoint: {SPACE_API_NAME}")
print("  expected signature (the Space must implement this):")
print("    fn(image: filepath, prompt: str, negative_prompt: str, num_frames: int, seed: int) -> video filepath")
print("  if your Space differs, adjust SPACE_API_NAME / render_job accordingly.")

In [ ]:
#@title 5 · Worker loop — claims jobs, calls the Space, uploads MP4s. Keep running.
import io, time, os, gc, tempfile, shutil
from PIL import Image

HEARTBEAT_SECONDS = 60
CLAIM_INTERVAL_SECONDS = 15
MAX_RENDER_SECONDS = 40 * 60   # hard stop: a job must never hang the loop

class JobCancelled(Exception):
    """Raised when the user cancels the job in the studio."""
    pass

def report_progress(key, status, progress, message):
    try:
        api("progress", {"workerJobKey": key, "status": status,
                         "progress": progress, "message": message})
    except Exception as e:
        print("  (progress update failed, continuing):", e)

def check_cancelled(key):
    try:
        out = api("job-status", {"workerJobKey": key})
        return bool(out.get("cancelled"))
    except Exception:
        return False

def upload_bytes(upload_url, data: bytes, content_type: str) -> str:
    r = requests.post(upload_url, data=data,
                      headers={"Content-Type": content_type}, timeout=600)
    r.raise_for_status()
    return r.json()["storageId"]

def readable_error(e: Exception) -> str:
    """Maps common failures to short, human-readable messages."""
    s = str(e)
    low = s.lower()
    if "timed out" in low or "timeout" in low or "time out" in low:
        return "The inference Space timed out (ZeroGPU queue or limits). Retry the job, or use fewer frames."
    if "quota" in low or "gpu limit" in low or "sleeping" in low:
        return "The inference Space hit its GPU quota or is sleeping. Try again later or duplicate the Space on your own account."
    if "401" in s or "unauthorized" in low or "denied" in low:
        return "Access to the inference Space was denied. Add a free HF_TOKEN in cell 1."
    if "image" in low and ("download" in low or "fetch" in low or "url" in low):
        return "Source image could not be downloaded from storage. Try uploading it again."
    if "videoUploadUrl" in s or "storageId" in s:
        return "Upload of the finished MP4 failed (storage upload URL rejected it)."
    return s[:400]

# The Wan pipeline requires height/width divisible by 16; 32 is the quality
# optimum but not required, so we snap to the least restrictive valid: 16.
def frame_size(aspect: str, resolution: str):
    short = 480 if resolution == "480p" else 720
    def snap(n): return int(round(n / 16)) * 16
    if aspect == "9:16":  return snap(short), snap(short * 16 / 9)
    if aspect == "1:1":   return snap(short), snap(short)
    return snap(short * 16 / 9), snap(short)  # 16:9 default

# Frames must satisfy (n - 1) % 4 == 0 for the Wan VAE (16 fps output).
def frames_for(duration_seconds: int) -> int:
    return 33 if duration_seconds <= 2 else 81

def render_job(job) -> bytes:
    """Delegates REAL image-to-video inference to the HF Space. No fakes."""
    s = job["settings"]
    resolution = s.get("resolution", "480p")
    if resolution == "720p":
        print("  (720p requested — clamped to 480p for free Space limits)")
        resolution = "480p"
    width, height = frame_size(s.get("aspectRatio", "16:9"), resolution)
    duration = int(s.get("durationSeconds", 5))
    num_frames = frames_for(duration)
    if duration > 5:
        print(f"  (duration {duration}s requested — clamped to ~5s / {num_frames} frames for Space limits)")

    # 1 · download the source image and prepare it as the first frame
    report_progress(job["workerJobKey"], "connecting", 5, "downloading source image")
    img_bytes = requests.get(job["imageUrl"], timeout=120).content
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img = img.resize((width, height), Image.LANCZOS)

    workdir = tempfile.mkdtemp(prefix="ims-")
    try:
        img_path = os.path.join(workdir, "source.png")
        img.save(img_path, format="PNG")

        # 2 · call the Space — this is where the actual WAN inference runs
        report_progress(job["workerJobKey"], "generating", 15, "queued on the inference Space")
        seed = int(s.get("seed") or 0)
        result = client.predict(
            image=handle_file(img_path),
            prompt=job["prompt"],
            negative_prompt=job.get("negativePrompt") or "",
            num_frames=num_frames,
            seed=seed,
            api_name=SPACE_API_NAME,
        )

        if check_cancelled(job["workerJobKey"]):
            raise JobCancelled()

        # 3 · the Space returns a filepath (or list/dict) for the MP4
        video_path = result
        if isinstance(video_path, (list, tuple)):
            video_path = video_path[0]
        if isinstance(video_path, dict):
            video_path = video_path.get("video") or video_path.get("path") or next(iter(video_path.values()))
        if not video_path or not os.path.exists(video_path):
            raise RuntimeError(f"The Space returned no video file: {result!r}")

        # 4 · encode happens on the Space; we read the real MP4 bytes back
        report_progress(job["workerJobKey"], "processing", 90, "uploading MP4")
        with open(video_path, "rb") as f:
            mp4 = f.read()
        if len(mp4) < 1024:
            raise RuntimeError("The Space returned an empty/invalid video file.")
        return mp4
    finally:
        shutil.rmtree(workdir, ignore_errors=True)
        gc.collect()

print("worker running — claim/heartbeat loop started (CPU mode, inference via HF Space)")
print("queue a job in the studio; this cell picks it up automatically")
print("stop the cell (⏹) or Runtime → Interrupt to take the worker offline")

last_heartbeat = 0.0
while True:
    now = time.time()
    try:
        if now - last_heartbeat > HEARTBEAT_SECONDS:
            api("health", {"status": "online", "gpuName": WORKER_LABEL, "workerVersion": WORKER_VERSION,
                           "message": "polling for jobs (CPU worker)"})
            last_heartbeat = now

        claim = api("claim", {})
        job = claim.get("job")
        if not job:
            time.sleep(CLAIM_INTERVAL_SECONDS)
            continue

        print("▸ job", job["workerJobKey"][:8], "·", job["model"])

        if check_cancelled(job["workerJobKey"]):
            print("  job was cancelled before render — skipping")
            continue

        started = time.time()
        try:
            mp4 = render_job(job)

            if time.time() - started > MAX_RENDER_SECONDS:
                raise TimeoutError("Render exceeded the hard time limit.")

            if check_cancelled(job["workerJobKey"]):
                print("  job was cancelled during render — discarding")
                continue

            video_storage_id = upload_bytes(job["videoUploadUrl"], mp4, "video/mp4")
            out = api("complete", {"workerJobKey": job["workerJobKey"],
                                   "videoStorageId": video_storage_id})
            print("  ✓ completed — videoId", out.get("videoId", "?"))

        except KeyboardInterrupt:
            raise
        except JobCancelled:
            print("  job was cancelled mid-render — discarding")
        except Exception as e:
            msg = readable_error(e)
            print("  ✗ job failed:", msg)
            try:
                api("fail", {"workerJobKey": job["workerJobKey"], "error": msg})
            except Exception as e2:
                print("  (fail report also failed):", e2)

    except KeyboardInterrupt:
        print("worker stopped by user")
        try:
            api("health", {"status": "error", "message": "worker stopped"})
        except Exception:
            pass
        break
    except Exception as e:
        # transient network errors: wait and retry, the studio shows the truth
        print("loop error (retrying):", str(e)[:200])
        time.sleep(10)